# UltraDES → ESP32 Supervisor Generator

Run all cells top-to-bottom. The last cell writes **`supervisor_data.json`** and **`supervisor_data.h`**.

- Copy `supervisor_data.json` to the root of the ESP32 SD card (SD-card firmware).
- Copy `supervisor_data.h` to your Arduino sketch folder (flash-embedded firmware).

Both files contain supervisors computed by **UltraDES** (genuine supC):
- `monolithic` — one supervisor for the entire system
- `local_modular` — one supervisor per specification (or per conflict-free group)

**Available problems** (set `PROBLEM` in Cell 3):
- `small_factory` — M1 → Buffer → M2
- `extended_small_factory` — M1 → B1 → M2 → B2 → M3
- `fms` — Flexible Manufacturing System (Queiroz & Cury 2000)

> **Requires UltraDES-Python.** Cell 1 installs it automatically on Linux/Colab.

In [8]:
# ── Cell 1: Install UltraDES ──────────────────────────────────────────────────
from IPython.display import clear_output

!sudo apt-get install -y dotnet-runtime-8.0
!pip install --no-cache-dir https://github.com/lacsed/UltraDES-Python/releases/download/0.0.6/ultrades_python-0.0.6-py3-none-any.whl

clear_output()

In [9]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import sys, os, json

from pythonnet import load
load("coreclr")

import ultrades
sys.path.append(os.path.dirname(ultrades.__file__))

from ultrades.automata import (
    state,
    event,
    dfa,
    monolithic_supervisor,
    local_modular_supervisors,
    parallel_composition,
)

print("UltraDES imported successfully.")

UltraDES imported successfully.


In [10]:
# ── Cell 3: Choose problem ────────────────────────────────────────────────────
# Change this line to switch problems:
#   'small_factory'           M1 -> Buffer -> M2
#   'extended_small_factory'  M1 -> B1 -> M2 -> B2 -> M3
#   'fms'                     Flexible Manufacturing System (Queiroz & Cury 2000)

PROBLEM = 'fms'

print(f'Problem selected: {PROBLEM}')

Problem selected: fms


In [11]:
# ── Cell 4: Problem definitions ───────────────────────────────────────────────
#
# Each builder returns (plants, specs, spec_groups, sim_seq):
#   plants      – list of plant DFAs
#   specs       – list of specification DFAs (all specs, for monolithic)
#   spec_groups – list of lists: each inner list is a conflict-free group of specs
#                 used for local modular synthesis.  For small/extended problems
#                 every spec is its own group (they share no events).
#                 For the FMS, specs that share events are grouped together so
#                 that local_modular_supervisors() does not raise ConflictingSupervisors.
#   sim_seq     – list of event name strings for the simulation sequence
#
# NOTE: UltraDES matches events by name string, not by object identity.
# Create separate event objects with the same name for plant and spec —
# they are logically the same event and UltraDES will synchronise on them.

# ─── Small Factory ────────────────────────────────────────────────────────────
# Plants : M1 (e1 start / e2 finish), M2 (e3 start / e4 finish)
# Spec   : Buf — buffer between M1 output and M2 input
#           (Buf_empty --e2--> Buf_full --e3--> Buf_empty)
# Controllable events: e1 (start M1), e3 (start M2)
# Conflict analysis  : only 1 spec → no conflict possible

def build_small_factory():
    # ── States ──
    m1_idle  = state('M1_idle',  marked=True)
    m1_busy  = state('M1_busy',  marked=False)
    m2_idle  = state('M2_idle',  marked=True)
    m2_busy  = state('M2_busy',  marked=False)
    buf_empty = state('Buf_empty', marked=True)
    buf_full  = state('Buf_full',  marked=False)

    # ── Plant events ──
    e1 = event('e1', controllable=True)   # M1 start
    e2 = event('e2', controllable=False)  # M1 finish
    e3 = event('e3', controllable=True)   # M2 start
    e4 = event('e4', controllable=False)  # M2 finish

    # ── Spec events (same names — UltraDES synchronises by name) ──
    e2_s = event('e2', controllable=False)
    e3_s = event('e3', controllable=True)

    # ── Plant automata ──
    M1  = dfa([(m1_idle, e1,   m1_busy),
               (m1_busy, e2,   m1_idle)], m1_idle, 'M1')
    M2  = dfa([(m2_idle, e3,   m2_busy),
               (m2_busy, e4,   m2_idle)], m2_idle, 'M2')

    # ── Specification automaton ──
    Buf = dfa([(buf_empty, e2_s, buf_full),
               (buf_full,  e3_s, buf_empty)], buf_empty, 'Buf')

    plants      = [M1, M2]
    specs       = [Buf]
    spec_groups = [[Buf]]           # 1 spec, 1 group
    sim_seq     = ['e1', 'e2', 'e3', 'e4']

    return plants, specs, spec_groups, sim_seq


# ─── Extended Small Factory ───────────────────────────────────────────────────
# Plants : M1 (a1/b1), M2 (a2/b2), M3 (a3/b3)
# Specs  : B1 — buffer between M1 and M2  (b1 fills, a2 empties)
#          B2 — buffer between M2 and M3  (b2 fills, a3 empties)
# Controllable events: a1, a2, a3
# Conflict analysis  : B1 uses {b1,a2}, B2 uses {b2,a3} → disjoint → no conflict

def build_extended_small_factory():
    # ── States ──
    m1i = state('M1_idle', True);  m1b = state('M1_busy', False)
    m2i = state('M2_idle', True);  m2b = state('M2_busy', False)
    m3i = state('M3_idle', True);  m3b = state('M3_busy', False)
    b1e = state('B1_empty', True); b1f = state('B1_full', False)
    b2e = state('B2_empty', True); b2f = state('B2_full', False)

    # ── Plant events ──
    a1 = event('a1', controllable=True);  b1 = event('b1', controllable=False)
    a2 = event('a2', controllable=True);  b2 = event('b2', controllable=False)
    a3 = event('a3', controllable=True);  b3 = event('b3', controllable=False)

    # ── Spec events ──
    b1_s = event('b1', controllable=False); a2_s = event('a2', controllable=True)
    b2_s = event('b2', controllable=False); a3_s = event('a3', controllable=True)

    # ── Plant automata ──
    M1 = dfa([(m1i, a1, m1b), (m1b, b1, m1i)], m1i, 'M1')
    M2 = dfa([(m2i, a2, m2b), (m2b, b2, m2i)], m2i, 'M2')
    M3 = dfa([(m3i, a3, m3b), (m3b, b3, m3i)], m3i, 'M3')

    # ── Specification automata ──
    B1 = dfa([(b1e, b1_s, b1f), (b1f, a2_s, b1e)], b1e, 'B1')
    B2 = dfa([(b2e, b2_s, b2f), (b2f, a3_s, b2e)], b2e, 'B2')

    plants      = [M1, M2, M3]
    specs       = [B1, B2]
    spec_groups = [[B1], [B2]]      # disjoint alphabets → each is its own group
    sim_seq     = ['a1', 'b1', 'a2', 'b2', 'a3', 'b3']

    return plants, specs, spec_groups, sim_seq


# ─── FMS — Flexible Manufacturing System (Queiroz & Cury 2000) ───────────────
# Plants : C1, C2, Lathe, Mill, Robot, AM, C3, PD
# Specs  : E1 … E8

def build_fms():
    def s(name, marked=True):
        return state(name, marked)

    def e(name, ctrl):
        return event(name, controllable=ctrl)

    # ===================== PLANTS =====================
    c1_0=s('c1_0'); c1_1=s('c1_1',False)
    C1 = dfa([(c1_0,e('11',True),c1_1),(c1_1,e('12',False),c1_0)], c1_0,'C1')

    c2_0=s('c2_0'); c2_1=s('c2_1',False)
    C2 = dfa([(c2_0,e('21',True),c2_1),(c2_1,e('22',False),c2_0)], c2_0,'C2')

    la_0=s('la_0'); la_1=s('la_1',False)
    Lathe = dfa([(la_0,e('41',True),la_1),(la_1,e('42',False),la_0)], la_0,'Lathe')

    pd_0=s('pd_0'); pd_1=s('pd_1',False)
    PD = dfa([(pd_0,e('81',True),pd_1),(pd_1,e('82',False),pd_0)], pd_0,'PD')

    mi_0=s('mi_0'); mi_1=s('mi_1',False); mi_2=s('mi_2',False)
    Mill = dfa([
        (mi_0,e('51',True),mi_1),(mi_1,e('52',False),mi_0),
        (mi_0,e('53',True),mi_2),(mi_2,e('54',False),mi_0)
    ], mi_0,'Mill')

    c3_0=s('c3_0'); c3_1=s('c3_1',False); c3_2=s('c3_2',False)
    C3 = dfa([
        (c3_0,e('71',True),c3_1),(c3_1,e('72',False),c3_0),
        (c3_0,e('73',True),c3_2),(c3_2,e('74',False),c3_0)
    ], c3_0,'C3')

    r0=s('r0'); r1=s('r1',False); r2=s('r2',False)
    r3=s('r3',False); r4=s('r4',False); r5=s('r5',False)
    Robot = dfa([
        (r0,e('31',True),r1),(r1,e('32',False),r0),
        (r0,e('33',True),r2),(r2,e('34',False),r0),
        (r0,e('35',True),r3),(r3,e('36',False),r0),
        (r0,e('37',True),r4),(r4,e('38',False),r0),
        (r0,e('39',True),r5),(r5,e('30',False),r0)
    ], r0,'Robot')

    # AM (Assembly Machine): 4 estados, conforme Figura 28 da tese
    a0=s('a0'); a1=s('a1',False); a2=s('a2',False); a3=s('a3',False)
    AM = dfa([
        (a0,e('61',True), a1),
        (a1,e('63',True), a2),(a2,e('64',False),a0),
        (a1,e('65',True), a3),(a3,e('66',False),a0)
    ], a0,'AM')

    plants = [C1,C2,Lathe,Mill,Robot,AM,C3,PD]

    # ===================== SPECS =====================
    E1 = dfa([(s('E1_0'),e('12',False),s('E1_1',False)),
              (s('E1_1',False),e('31',True),s('E1_0'))], s('E1_0'), 'E1')

    E2 = dfa([(s('E2_0'),e('22',False),s('E2_1',False)),
              (s('E2_1',False),e('33',True),s('E2_0'))], s('E2_0'), 'E2')

    e3_0=s('E3_0'); e3_1=s('E3_1',False); e3_2=s('E3_2',False)
    E3 = dfa([
        (e3_0,e('32',False),e3_1),(e3_1,e('41',True),e3_0),
        (e3_0,e('42',False),e3_2),(e3_2,e('35',True),e3_0),
    ], e3_0,'E3')

    e4_0=s('E4_0'); e4_1=s('E4_1',False)
    e4_2=s('E4_2',False); e4_3=s('E4_3',False)
    E4 = dfa([
        (e4_0,e('34',False),e4_1),
        (e4_1,e('51',True),e4_0),(e4_1,e('53',True),e4_0),
        (e4_0,e('52',False),e4_2),(e4_2,e('37',True),e4_0),
        (e4_0,e('54',False),e4_3),(e4_3,e('39',True),e4_0)
    ], e4_0,'E4')

    E5 = dfa([(s('E5_0'),e('36',False),s('E5_1',False)),
              (s('E5_1',False),e('61',True),s('E5_0'))], s('E5_0'), 'E5')

    E6 = dfa([(s('E6_0'),e('38',False),s('E6_1',False)),
              (s('E6_1',False),e('63',True),s('E6_0'))], s('E6_0'), 'E6')

    # ===== SPECS E7 e E8 — buffers B7 e B8 =====
    # Conforme Figura 27 e 29 da tese.
    # Eventos controlaveis (c): iniciam operacao de uma maquina
    # Eventos nao-controlaveis (nc): finalizam operacao de uma maquina
    #
    # E7: buffer B7
    #   caminho 1: Robot deposita (30=nc) -> C3 pega (71=c)
    #   caminho 2: C3 deposita (74=nc)    -> MM pega  (65=c)
    e7_0=s('E7_0'); e7_1=s('E7_1',False); e7_2=s('E7_2',False)
    E7 = dfa([
        (e7_0, e('30',False), e7_1), (e7_1, e('71',True), e7_0),
        (e7_0, e('74',False), e7_2), (e7_2, e('65',True), e7_0),
    ], e7_0, 'E7')

    # E8: buffer B8
    #   caminho 1: C3 deposita  (72=nc) -> PD pega   (81=c)
    #   caminho 2: PD deposita  (82=nc) -> C3 pega   (73=c)
    e8_0=s('E8_0'); e8_1=s('E8_1',False); e8_2=s('E8_2',False)
    E8 = dfa([
        (e8_0, e('72',False), e8_1), (e8_1, e('81',True), e8_0),
        (e8_0, e('82',False), e8_2), (e8_2, e('73',True), e8_0),
    ], e8_0, 'E8')

    # Composicao paralela E78 = E7 || E8 como especificacao unica
    # para resolver o conflito (conforme tese, Capitulo 6)
    E78 = parallel_composition(E7, E8)

    specs = [E1,E2,E3,E4,E5,E6,E78]

    spec_groups = [
        [E1],
        [E2],
        [E3],
        [E4],
        [E5],
        [E6],
        [E78]
    ]

    sim_seq = ['11','12','31','32','41','42','35','36','61',
               '39','30','71','72','81','82','73','74']

    return plants, specs, spec_groups, sim_seq


PROBLEMS = {
    'small_factory':          build_small_factory,
    'extended_small_factory': build_extended_small_factory,
    'fms':                    build_fms,
}

assert PROBLEM in PROBLEMS, (
    f'Unknown problem "{PROBLEM}". '
    f'Choose from: {list(PROBLEMS.keys())}'
)
print('Problem definitions loaded.')

Problem definitions loaded.


In [12]:
# ── Cell 5: Extraction helpers ────────────────────────────────────────────────
#
# _extract_ultrades  — converts a UltraDES supervisor DFA into the JSON dict
#                      expected by the ESP32 firmware.
#
# _global_events     — collects every event name from plants+specs using the
#                      PascalCase .Transitions attribute (UltraDES-Python wraps
#                      the .NET DFA; all properties are PascalCase).

def _global_events(plants, specs):
    """Return sorted list of all event names appearing in plants and specs."""
    evs = set()
    for automaton in plants + specs:
        for tr in automaton.Transitions:
            evs.add(str(tr.Trigger))
    return sorted(evs)


def _extract_ultrades(sup, global_events, name, method):
    """
    Convert a UltraDES DFA object into the dict format expected by the ESP32.

    Fields produced
    ---------------
    name          : label for this supervisor
    method        : 'monolithic' or 'local_modular'
    num_states    : total number of states
    initial_state : one-hot vector  (index of the initial state == 1)
    own_events    : sorted list of event names in this supervisor's alphabet
    transitions   : { event_name: [[origin_idx, dest_idx], ...], ... }
    enablement    : { event_name: [0/1 per state — 1 = enabled in that state] }
    nonblocking   : True  (UltraDES supC is always nonblocking)
    reachable     : True if the initial state was found in the transition set
    """
    state_set = set()
    trans_list = []  # lista de (origin, trigger, dest)

    for tr in sup.Transitions:
        origin  = str(tr.Origin)
        trigger = str(tr.Trigger)
        dest    = str(tr.Destination)
        state_set.update([origin, dest])
        trans_list.append((origin, trigger, dest))

    # State ordering: initial state first, rest sorted lexicographically
    init = str(sup.InitialState)
    sl = sorted(state_set, key=str)
    if init in sl:
        sl.remove(init)
        sl = [init] + sl

    sid = {s: i for i, s in enumerate(sl)}
    n   = len(sl)

    # Initial-state one-hot vector
    init_vec = [0] * n
    if init in sid:
        init_vec[sid[init]] = 1

    # Own events (local alphabet of this supervisor)
    own = sorted({trigger for (_, trigger, _) in trans_list})

    # Transitions dict: event -> list of [origin_idx, dest_idx]
    trans = {}
    for (orig, ev, dest) in trans_list:
        trans.setdefault(ev, []).append([sid[orig], sid[dest]])

    # Enablement matrix: event -> per-state 0/1 flag
    enablement = {}
    for ev in own:
        row = [0] * n
        for (orig, trigger, _) in trans_list:
            if trigger == ev:
                row[sid[orig]] = 1
        enablement[ev] = row

    return {
        'name':            name,
        'method':          method,
        'num_states':      n,
        'num_transitions': sum(len(v) for v in trans.values()),
        'initial_state':   init_vec,
        'own_events':      own,
        'transitions':     trans,
        'enablement':      enablement,
        'nonblocking':     True,
        'reachable':       (init in state_set),
    }


print('Extraction helpers ready.')

Extraction helpers ready.


In [13]:
# ── Cell 6: Supervisor synthesis ─────────────────────────────────────────────
#
# LOCAL MODULAR STRATEGY
# ──────────────────────
# local_modular_supervisors(plants, specs) raises "Conflicting Supervisors" when
# the specs are not pairwise non-conflicting — a necessary condition for the
# local modular approach.
#
# The solution: synthesise each CONFLICT-FREE GROUP independently.
# Each group produces one supervisor that is added to lmod_list.
# The spec_groups field returned by each problem builder encodes the grouping.
#
# For small_factory / extended_small_factory every spec is its own group
# (disjoint alphabets → trivially non-conflicting).
# For the FMS, specs that share events through the plant are grouped together.

import traceback

try:
    base_plants, base_specs, spec_groups, sim_seq = PROBLEMS[PROBLEM]()

    gevents = _global_events(base_plants, base_specs)
    print(f'Problem       : {PROBLEM}')
    print(f'Plants        : {[g.Name for g in base_plants]}')
    print(f'Specs         : {[e.Name for e in base_specs]}')
    print(f'Spec groups   : {[[s.Name for s in g] for g in spec_groups]}')
    print(f'Global events : {gevents}')
    print()

    # ── Monolithic supervisor ──────────────────────────────────────────────────
    # monolithic_supervisor(plants, specs) — computes the supremal controllable
    # sublanguage for the full plant ∥ spec composition in a single step.
    print('Computing monolithic supervisor …')
    mono_sup  = monolithic_supervisor(base_plants, base_specs)
    # SupC(K,G) = S||G — compose supervisor with plant to get full closed-loop
    plant_composition = base_plants[0]
    for p in base_plants[1:]:
        plant_composition = parallel_composition(plant_composition, p)
    mono_closed = parallel_composition(mono_sup, plant_composition)
    raw_closed = len(list(mono_closed.Transitions))
    raw_sup    = len(list(mono_sup.Transitions))
    mono_dict = _extract_ultrades(mono_sup, gevents, 'Monolithic', 'monolithic')
    print(f'  States (supervisor)         : {mono_dict["num_states"]}')
    print(f'  Transitions (supervisor)    : {raw_sup}')
    print(f'  Transitions (S||G)          : {raw_closed}')
    print(f'  Reachable initial state     : {mono_dict["reachable"]}')
    print()

    # ── Local modular supervisors ──────────────────────────────────────────────
    # Each conflict-free group is synthesised independently.
    # local_modular_supervisors(plants, group) returns one supervisor per spec
    # in the group; all are added to lmod_list.
    print('Computing local modular supervisors …')
    lmod_list = []

    sup_idx = 0
    for g_idx, group in enumerate(spec_groups):
        group_names = [s.Name for s in group]
        print(f'  Group {g_idx}: specs {group_names}')
        group_sups = local_modular_supervisors(base_plants, group)
        for sup in group_sups:
            label = f'S{sup_idx}'
            d = _extract_ultrades(sup, gevents, label, 'local_modular')
            lmod_list.append(d)
            print(f'    {label}: {d["num_states"]} states, {d["num_transitions"]} transitions  |  own events: {d["own_events"]}')
            sup_idx += 1

    print()
    print('✅ Synthesis complete.')
    print(f'   Monolithic : 1 supervisor, {mono_dict["num_states"]} states, {mono_dict["num_transitions"]} transitions')
    print(f'   Local mod  : {len(lmod_list)} supervisors, '
          f'{sum(d["num_states"] for d in lmod_list)} states total, '
          f'{sum(d["num_transitions"] for d in lmod_list)} transitions total')

except Exception as exc:
    print(f'❌ Synthesis error: {exc}')
    traceback.print_exc()

Problem       : fms
Plants        : ['C1', 'C2', 'Lathe', 'Mill', 'Robot', 'AM', 'C3', 'PD']
Specs         : ['E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7||E8']
Spec groups   : [['E1'], ['E2'], ['E3'], ['E4'], ['E5'], ['E6'], ['E7||E8']]
Global events : ['11', '12', '21', '22', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '41', '42', '51', '52', '53', '54', '61', '63', '64', '65', '66', '71', '72', '73', '74', '81', '82']

Computing monolithic supervisor …
  States (supervisor)         : 45504
  Transitions (supervisor)    : 200124
  Transitions (S||G)          : 200124
  Reachable initial state     : True

Computing local modular supervisors …
  Group 0: specs ['E1']
    S0: 18 states, 40 transitions  |  own events: ['11', '12', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39']
  Group 1: specs ['E2']
    S1: 18 states, 40 transitions  |  own events: ['21', '22', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39']
  Group 2: specs ['E3']
    S2: 18 states, 

In [15]:
# ── Cell 7: Write supervisor_data.json AND supervisor_data.h ─────────────────
#
# SIZE-AWARE EXPORT
# -----------------
# The ESP32 has ~1.5 MB of available program flash for the default partition
# scheme (2 MB sketch partition).  A raw-string literal in a .h file is part
# of the binary, so the compact JSON must fit within that budget.
#
# MONO_FLASH_LIMIT (bytes) — if the monolithic supervisor alone exceeds this
# the header is generated WITHOUT the monolithic entry and the .ino will
# automatically skip the monolithic benchmark and run only local modular.
#
# Typical sizes for the FMS problem:
#   monolithic  JSON contribution: ~5.4 MB  → does NOT fit
#   local_mod   JSON contribution: ~0.3 MB  → fits easily
#
# For small_factory / extended_small_factory the monolithic supervisor is tiny
# and both fit.  The limit below is conservative; adjust to your partition.

import json, os

OUTPUT_JSON = 'supervisor_data.json'
OUTPUT_H    = 'supervisor_data_' + PROBLEM + '.h'

MONO_FLASH_LIMIT = 1_400_000   # bytes — leave some headroom for code+stack

# ── Serialise each section independently so we can measure sizes ─────────────
mono_json_bytes  = len(json.dumps(mono_dict,  separators=(',', ':')).encode())
lmod_json_bytes  = len(json.dumps(lmod_list,  separators=(',', ':')).encode())
meta_json_bytes  = len(json.dumps({
    'problem':             PROBLEM,
    'global_event_names':  gevents,
    'simulation_sequence': sim_seq,
}, separators=(',', ':')).encode())

print(f'JSON size estimate:')
print(f'  metadata      : {meta_json_bytes:>10,} bytes')
print(f'  monolithic    : {mono_json_bytes:>10,} bytes')
print(f'  local_modular : {lmod_json_bytes:>10,} bytes')
print(f'  MONO_FLASH_LIMIT = {MONO_FLASH_LIMIT:,} bytes')
print()

include_mono = (mono_json_bytes <= MONO_FLASH_LIMIT)
if include_mono:
    print('✅  Monolithic fits — including BOTH supervisors in the header.')
else:
    print('⚠️   Monolithic is TOO LARGE for flash.')
    print('    The header will contain ONLY the local modular supervisors.')
    print('    The .ino will detect the missing monolithic entry and skip it automatically.')
print()

# ── Build payload ─────────────────────────────────────────────────────────────
payload = {
    'problem':             PROBLEM,
    'global_event_names':  gevents,
    'simulation_sequence': sim_seq,
    'local_modular':       lmod_list,
}
if include_mono:
    payload['monolithic'] = mono_dict

# ── JSON ──────────────────────────────────────────────────────────────────────
with open(OUTPUT_JSON, 'w') as f:
    json.dump(payload, f, indent=2)
print(f'Written: {OUTPUT_JSON}  ({os.path.getsize(OUTPUT_JSON):,} bytes)')

# ── C header ──────────────────────────────────────────────────────────────────
json_compact = json.dumps(payload, separators=(',', ':'))
total_h_bytes = len(json_compact.encode())

mono_status = 'INCLUDED' if include_mono else 'OMITTED (too large for flash — local modular only)'

h_content = (
    '// supervisor_data.h — auto-generated by the UltraDES notebook.\n'
    '// DO NOT EDIT BY HAND. Re-run Cell 7 to regenerate.\n'
    '//\n'
    f'// Problem    : {PROBLEM}\n'
    f'// Events     : {gevents}\n'
    f'// Sim seq    : {sim_seq}\n'
    f'// Monolithic : {mono_status}\n'
    f'// Header size: {total_h_bytes:,} bytes\n'
    '\n'
    '#pragma once\n'
    '\n'
    '// SUPERVISOR_JSON contains the full payload.\n'
    '// If the "monolithic" key is absent the .ino skips the monolithic\n'
    '// benchmark and runs only the local modular benchmark.\n'
    'static const char SUPERVISOR_JSON[] = R"JSON(\n'
    + json_compact + '\n'
    + ')JSON";\n'
)

with open(OUTPUT_H, 'w') as f:
    f.write(h_content)
print(f'Written: {OUTPUT_H}  ({os.path.getsize(OUTPUT_H):,} bytes)')
print()
print('Next steps:')
print(f'  1. Copy {OUTPUT_H} to your Arduino sketch folder (same folder as the .ino).')
print( '  2. Flash homomorphic-esp32-ultrades-json-nova.ino.')
if include_mono:
    print( '  Both MONOLITHIC and LOCAL MODULAR benchmarks run automatically.')
else:
    print( '  Only the LOCAL MODULAR benchmark runs (monolithic was too large for flash).')


JSON size estimate:
  metadata      :        304 bytes
  monolithic    :  5,625,416 bytes
  local_modular :     22,698 bytes
  MONO_FLASH_LIMIT = 1,400,000 bytes

⚠️   Monolithic is TOO LARGE for flash.
    The header will contain ONLY the local modular supervisors.
    The .ino will detect the missing monolithic entry and skip it automatically.

Written: supervisor_data.json  (140,311 bytes)
Written: supervisor_data_fms.h  (23,820 bytes)

Next steps:
  1. Copy supervisor_data_fms.h to your Arduino sketch folder (same folder as the .ino).
  2. Flash homomorphic-esp32-ultrades-json-nova.ino.
  Only the LOCAL MODULAR benchmark runs (monolithic was too large for flash).
